# News Embeddings Pipeline — Step by Step

Braço 02: the frozen text corpus → one dense vector per article from a **bracket**
of six encoders → mean per (epidemiological week, location).

Mirrors `src/pipeline/news/embeddings/` (spec → steps → transform → integrity).
The hypothesis, the evidence and the counter-evidence are in
[`notes/02-embeddings-densos.md`](../notes/02-embeddings-densos.md); the methods
write-up and the measured numbers are in
[`src/pipeline/news/README.md`](../src/pipeline/news/README.md) §3.

This is one of three **competing representations** of the same corpus — not a
layer on top of braço 01. All three read the same frozen table and publish the
same grain, so the comparison isolates the representation and nothing else. See
[`notes/00-protocolo-comparacao.md`](../notes/00-protocolo-comparacao.md).

**Braço 02 is a bracket, not an encoder.** One dense configuration makes a null
result uninterpretable: "the representation lost" cannot be told apart from "this
particular encoder was too weak". Six unsupervised, untrained configurations are
published and read as a range.

## Setup

In [1]:
import sys
from pathlib import Path

import numpy as np
import polars as pl

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from pipeline.news.embeddings import steps as emb_steps
from pipeline.news.embeddings.integrity import run_all as embedding_checks
from pipeline.news.embeddings.spec import (
    ARTICLE_ID, BASE_VARIANT_ID, EMB_MEAN, EMBEDDING, EMBEDDING_DIM, EW, LOCATION,
    N_ARTICLES, N_TOKENS, VARIANT, VARIANTS, VARIANT_ORDER, table_names,
)
from pipeline.news.embeddings.transform import select_articles
from pipeline.news.lexicon import steps as lex_steps
from pipeline.news.text import steps as text_steps

pl.Config.set_tbl_rows(16)
pl.Config.set_fmt_str_lengths(70)

FIXTURES = PROJECT_ROOT / "tests" / "fixtures" / "news"
LEXICON_PATH = PROJECT_ROOT / "data" / "news" / "reference" / "lexico_dengue.yaml"
FU_PATH = PROJECT_ROOT / "data" / "epidemiological" / "br_federative_units.csv"

NEWS_DIR = PROJECT_ROOT / "data" / "pipeline" / "news"
CORPUS_PATH = NEWS_DIR / "EBC_dengue_text.parquet"
SINAN_PATH = PROJECT_ROOT / "data" / "epidemiological" / "SINAN" / "SINAN_dengue_cases.parquet"
LEXICON_EW_PATH = NEWS_DIR / "EBC_dengue_lexicon_EW.parquet"      # braço 01
TOPICS_EW_PATH = NEWS_DIR / "EBC_dengue_topics_EW.parquet"        # braço 03


def tables_for(variant_id: str) -> tuple[Path, Path]:
    # (article table, pooled table) on disk for one configuration.
    article, pooled = table_names(variant_id)
    return NEWS_DIR / f"{article}.parquet", NEWS_DIR / f"{pooled}.parquet"


def bracket_ready() -> bool:
    return all(a.exists() and p.exists() for a, p in map(tables_for, VARIANT_ORDER))


FIXTURES.exists(), LEXICON_PATH.exists(), FU_PATH.exists(), bracket_ready()

(True, True, True, True)

## What this notebook has to prove

1. The bracket is a **range of unsupervised configurations**, each with its encoder
   pinned to a commit sha, so "which BERTimbau?" has an answer that does not move.
2. Pooling is **per variant** and matches what each encoder's authors published:
   masked mean for BERTimbau, E5 and Serafim, `[CLS]` for BGE-M3.
3. Padding does not change a vector — the mask is what makes the batch irrelevant.
4. The token window is **512 for every model that can take it**, with Serafim's 128
   as a declared exception, so a gain cannot come from one encoder reading more.
5. Whitening is the documented fix for the anisotropy of unfine-tuned mean pooling,
   and it changes the geometry without changing the aggregation contract.
6. The article set is the **same** one braço 01 sees: the language and dengue-term
   gates are common ground under protocol 00, not representation.
7. A week with no article is **absent**, not a zero vector.
8. `emb_std` is **gone**, and the notebook has to show why: for unit vectors it was
   an exact function of `emb_mean`, so it carried no information the mean did not.
9. The fixture is five pages, so it cannot show truncation at scale, the weekly
   coverage or the Spanish slice. Those need the real corpus, and the **Optional —
   the real corpus** section below is where they are measured.

## Step 1 — the bracket

Six configurations, and the registry in `spec.VARIANTS` is the only place they are
named. The config file carries a single `variant:` key rather than one key per
parameter, because these *are* the design being compared: a config key that
silently overrode the pooling would make a published table describe a configuration
written down nowhere.

In [2]:
rows = []
for variant_id in VARIANT_ORDER:
    settings = VARIANTS[variant_id]
    rows.append({
        "variant": settings.id,
        "encoder": settings.model_name.split("/")[-1],
        "revision": settings.model_revision[:8],
        "pooling": settings.pooling,
        "window": settings.max_tokens,
        "input": settings.input_fields,
        "prefix": repr(settings.prefix),
        "whitening": settings.whitening,
    })
print(pl.DataFrame(rows))
print()
for variant_id in VARIANT_ORDER:
    settings = VARIANTS[variant_id]
    print(f"* {settings.label} — {settings.note.split('.')[0]}.")

shape: (6, 8)
┌──────────────┬──────────────┬──────────┬─────────┬────────┬────────────┬─────────────┬───────────┐
│ variant      ┆ encoder      ┆ revision ┆ pooling ┆ window ┆ input      ┆ prefix      ┆ whitening │
│ ---          ┆ ---          ┆ ---      ┆ ---     ┆ ---    ┆ ---        ┆ ---         ┆ ---       │
│ str          ┆ str          ┆ str      ┆ str     ┆ i64    ┆ str        ┆ str         ┆ bool      │
╞══════════════╪══════════════╪══════════╪═════════╪════════╪════════════╪═════════════╪═══════════╡
│ bertimbau-me ┆ bert-base-po ┆ 94d69c95 ┆ mean    ┆ 512    ┆ title_body ┆ ''          ┆ false     │
│ an           ┆ rtuguese-cas ┆          ┆         ┆        ┆            ┆             ┆           │
│              ┆ ed           ┆          ┆         ┆        ┆            ┆             ┆           │
│ bertimbau-wh ┆ bert-base-po ┆ 94d69c95 ┆ mean    ┆ 512    ┆ title_body ┆ ''          ┆ true      │
│ itening      ┆ rtuguese-cas ┆          ┆         ┆        ┆            ┆   

Two entries in that table are decisions rather than conveniences:

- **BGE-M3 is read with `[CLS]`** because that *is* its dense embedding — its
  `1_Pooling/config.json` sets `pooling_mode_cls_token: true`. Forcing a mean on it
  would measure a configuration nobody uses, and a weak BGE-M3 would be
  uninterpretable — the exact failure the bracket exists to avoid.
- **E5 is prefixed with `passage: `** because it is trained with instruction
  prefixes, and these are articles rather than queries.

Two risks are declared rather than hidden: E5-large and BGE-M3 do not document their
pretraining cutoff, and protocol 00 asks for open weights *with a known cutoff*; and
whitening is fitted on the corpus, so a fit over the whole corpus has seen the
period being predicted. `transform(..., whitening_fit_until=...)` is the argument
that fixes the second one when the rolling-origin evaluation runs.

## Step 2 — the frozen encoder

`load_encoder` names the *revision* as well as the model. A branch moves under the
artifact, and an embedding computed last month would stop being reproducible.

It needs the optional extra, which is deliberately not a core dependency —
`torch` is ~800 MB and nothing else in the repository imports it:

```bash
uv sync --extra embeddings
```

The cell below falls back to a deterministic stand-in when the extra is missing,
so the aggregation steps still run. **When it falls back, the vectors are not
BERTimbau** and nothing downstream of them is a result.

In [3]:
settings = VARIANTS[BASE_VARIANT_ID]
try:
    encoder = emb_steps.load_encoder(
        settings.model_name,
        settings.model_revision,
        pooling=settings.pooling,
        prefix=settings.prefix,
        max_tokens=settings.max_tokens,
    )
    print(f"loaded {encoder.model_id}@{encoder.model_revision[:8]} — {encoder.dim}-wide, "
          f"{encoder.pooling} pooling, {encoder.max_tokens}-token window")
except emb_steps.MissingExtraError as error:
    import zlib

    class _StandIn:
        # Deterministic text -> unit vector. NOT BERTimbau.
        model_id = settings.model_name
        model_revision = settings.model_revision
        dim = EMBEDDING_DIM
        pooling = settings.pooling
        max_tokens = settings.max_tokens

        def encode(self, texts):
            vectors = []
            for text in texts:
                rng = np.random.default_rng(zlib.crc32(text.encode("utf-8")))
                vector = rng.standard_normal(self.dim)
                vectors.append(vector / np.linalg.norm(vector))
            return emb_steps.Encoded(
                vectors=np.stack(vectors).astype(np.float32),
                n_tokens=[len(t.split()) for t in texts],
            )

    encoder = _StandIn()
    print(error)
    print(">>> FALLBACK ACTIVE: the vectors below are NOT BERTimbau <<<")

/Users/caio.nunes/Projects/arbovirus-surveillance-data/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 43812.21it/s]


[transformers] BertModel LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


loaded neuralmind/bert-base-portuguese-cased@94d69c95 — 768-wide, mean pooling, 512-token window


## Step 3 — pooling, masked

Three ways to turn a `(tokens, 768)` matrix into a `(768,)` vector, and the choice
is not cosmetic:

| read | what it is | where it is used |
|---|---|---|
| `[CLS]` | the token BERT trains for next-sentence prediction | **BGE-M3**, whose dense embedding is defined that way |
| pooler head | a linear layer on top of `[CLS]` | nowhere — discarded by every sentence-embedding recipe |
| **masked mean** | the average over the real tokens | **BERTimbau, E5-large, Serafim** |

"Masked" is the load-bearing word. A padded batch carries `[PAD]` positions, and
averaging over them mixes in whatever the model happens to represent `[PAD]` as —
so the same sentence would get a different vector depending on what it was batched
with. The check below encodes one sentence alone and again next to a shorter one:
the vectors must be identical.

In [4]:
probe = "A dengue avança no interior de São Paulo e a rede pública alerta para o surto."
alone = encoder.encode([probe]).vectors[0]
padded = encoder.encode([probe, "Nota curta."]).vectors[0]

print("same text, alone vs padded in a batch of 2")
print(f"  max |difference|   = {np.abs(alone - padded).max():.2e}")
print(f"  norm of the vector = {np.linalg.norm(alone):.6f}   (L2-normalized)")
print(f"  pooling            = {settings.pooling}, {settings.max_tokens}-token window")

same text, alone vs padded in a batch of 2
  max |difference|   = 0.00e+00
  norm of the vector = 1.000000   (L2-normalized)
  pooling            = mean, 512-token window


## Step 4 — whitening, and the anisotropy it corrects

A vanilla encoder is anisotropic: token vectors crowd into a narrow cone, so
unrelated sentences look alike
([Ethayarajh 2019](https://doi.org/10.18653/v1/D19-1006)). Mean pooling without
fine-tuning inherits that, and it is the one documented weakness of the base
configuration.

The fix that needs no labels and no training is a linear map fitted on the vectors
themselves: centre them, then multiply by `U√Λ⁻¹` from the SVD of their covariance
([Su 2021](https://arxiv.org/abs/2103.15316);
[WhiteningBERT](https://doi.org/10.18653/v1/2021.findings-emnlp.23)).

Two details are not incidental. The eigenvalues are **floored** — the covariance of
768 unit vectors is near-singular, so the smallest eigenvalues are numerical noise
and their exact reciprocals are not a transform. And the output is **re-normalized**:
the base configuration pools unit vectors, so a whitened variant that pooled
magnitude-weighted means would differ from the base in two ways at once.

The demo below is on the five fixture pages, which is a thin sample for a
covariance — the real measurement is in the *Optional* section.

In [5]:
raw = np.random.default_rng(0).standard_normal((300, 16)) * 0.05
raw[:, 0] += 1.0
cone = emb_steps.normalize_rows(raw)
params = emb_steps.fit_whitening(cone)
spread = emb_steps.apply_whitening(cone, params)


def mean_cosine(matrix):
    unit = emb_steps.normalize_rows(matrix)
    gram = unit @ unit.T
    n = gram.shape[0]
    return float((gram.sum() - np.trace(gram)) / (n * (n - 1)))


print("a synthetic narrow cone, 300 vectors in 16 dims")
print(f"  mean pairwise cosine, before whitening = {mean_cosine(cone):+.4f}")
print(f"  mean pairwise cosine, after whitening  = {mean_cosine(spread):+.4f}")
print(f"  every output row is still a unit vector: "
      f"{np.allclose(np.linalg.norm(spread, axis=1), 1.0, atol=1e-5)}")
print(f"  floor on the eigenvalues = {emb_steps.WHITENING_EPSILON} "
      f"(without it the near-singular directions invert into noise)")

a synthetic narrow cone, 300 vectors in 16 dims
  mean pairwise cosine, before whitening = +0.9638
  mean pairwise cosine, after whitening  = -0.0020
  every output row is still a unit vector: True
  floor on the eigenvalues = 1e-05 (without it the near-singular directions invert into noise)


## Step 5 — one vector per article

The article set is decided *before* the encoder runs, by the same two gates braço
01 applies: drop the ~3.7% of the corpus that is Spanish (Agência Brasil publishes
a Spanish edition), and drop pages that never name a dengue-family disease.

This is not a detail. If one braço sees 4,369 articles and another sees 4,569, the
comparison measures the filter rather than the representation, which is exactly
what protocol 00 exists to prevent.

In [6]:
corpus = text_steps.build(FIXTURES)
lexicon = lex_steps.load_lexicon(LEXICON_PATH)
matcher = lex_steps.load_uf_matcher(FU_PATH)

kept, dropped = select_articles(corpus, lexicon)
articles = emb_steps.encode_articles(kept, encoder, batch_size=2, variant_id=settings.id)

print(f"{corpus.height} fixture pages → {kept.height} kept, {dropped} dropped by the shared gates")
print(f"article table: {articles.height} rows, embedding column {articles.schema[EMBEDDING]}\n")
articles.select(ARTICLE_ID, EW, VARIANT, N_TOKENS, "dim").head()

5 fixture pages → 5 kept, 0 dropped by the shared gates
article table: 5 rows, embedding column Array(Float32, shape=(768,))



article_id,ew,variant,n_tokens,dim
str,date,str,i64,i64
"""2002-09-30_584838_b533c1""",2002-09-29,"""bertimbau-mean""",68,768
"""2008-04-02_escolas-precisam-realizar-trabalho-de-conscientizacao-sobre…",2008-03-30,"""bertimbau-mean""",28,768
"""2011-01-11_ministros-discutem-acoes-de-prevencao-e-controle-da-dengue-…",2011-01-09,"""bertimbau-mean""",141,768
"""2016-02-04_presidente-dilma-pede-uniao-no-combate-ao-aedes-aegypti_1ce…",2016-01-31,"""bertimbau-mean""",57,768
"""2021-03-14_synthetic-fallback-page_000001""",2021-03-14,"""bertimbau-mean""",141,768


### Truncation is recorded, not hidden

`n_tokens` is the length the tokenizer produced *before* the window cut. It is
carried per row because a 34,960-character era-B article and a 163-character era-C
note both become one vector, and only this column says which of them lost text.

The window is per variant, so the count has to be taken per variant — otherwise
`serafim-title`'s 128 tokens would be invisible in the run that used them.

The counts below come from the *base* encoder's tokenizer, which is what is loaded
in this section; the other tokenizers differ by a few tokens on the same text. The
real-corpus section counts each variant from its own table, which is the number
that belongs in the write-up.

In [7]:
for variant_id in VARIANT_ORDER:
    window = VARIANTS[variant_id].max_tokens
    over, longest = emb_steps.truncation_stats(articles, window)
    print(f"{variant_id:22s} window {window:4d} → {over}/{articles.height} fixture pages over, "
          f"longest {longest} tokens")

bertimbau-mean         window  512 → 0/5 fixture pages over, longest 141 tokens
bertimbau-whitening    window  512 → 0/5 fixture pages over, longest 141 tokens
e5-large               window  512 → 0/5 fixture pages over, longest 141 tokens
bge-m3                 window  512 → 0/5 fixture pages over, longest 141 tokens
title-only             window  512 → 0/5 fixture pages over, longest 141 tokens
serafim-title          window  128 → 2/5 fixture pages over, longest 141 tokens


## Step 6 — the shared scoping

`UfMatcher` is imported from braço 01 on purpose. News is national and the target
is a UF, so the spatial mismatch is a **declared** limitation; re-solving it per
braço would mean the three representations no longer differ only in how they read
the text.

Two consequences worth remembering while reading any per-UF row below: full state
names match case-sensitively (`Pará` vs the preposition `para`), and municipalities
are not resolved — an article that says only "Araçutuba" counts for BR and not for SP.

In [8]:
locations = emb_steps.scoped_locations(kept, matcher)
locations.head(6)

article_id,locations
str,list[str]
"""2002-09-30_584838_b533c1""","[""RJ""]"
"""2008-04-02_escolas-precisam-realizar-trabalho-de-conscientizacao-sobre…",[]
"""2011-01-11_ministros-discutem-acoes-de-prevencao-e-controle-da-dengue-…",[]
"""2016-02-04_presidente-dilma-pede-uniao-no-combate-ao-aedes-aegypti_1ce…",[]
"""2021-03-14_synthetic-fallback-page_000001""",[]


## Step 7 — pooling to (ew, location)

Every article counts for `BR`; it also counts for each UF it names. One article can
therefore land in several rows of the same week — the same rule braço 01 uses, and
the reason `n_articles` can sum to more than the size of the corpus.

**`emb_std` used to be here and was removed on 2026-10-14.** It was published as the
norm of the per-dimension standard deviation, on the argument that a 768-wide std
column would double the table to say one number the norm already carries. The
argument was right and the conclusion was wrong: for L2-normalized article vectors
the parallel-axis identity on the unit sphere gives

    ‖std‖² = (1/n) Σᵢ ‖vᵢ − m‖² = 1 − ‖m‖²

so the scalar was an **exact deterministic function of `emb_mean`**, a column the
table already carried. Measured on the real tables: maximum deviation 1.6e-06 over
the 2,073 rows with n > 1. The documented "undefined on 59.8% of rows" was the same
fact from the other side — one article gives ‖m‖ = 1 and therefore exactly 0.

No scalar dispersion of unit vectors escapes this: mean pairwise cosine distance is
1 − ‖m‖² too. So the column was deleted rather than repaired, and the dispersion
hypothesis is now read as absorbed by the mean.

A week with no article is **absent**, not a zero vector. A zero embedding is a real
point in the space and would read as "news that means nothing" rather than "no
news". `notes/02` asks for a `no_news` indicator; absence is that indicator, and
the consumer left-joins onto the target calendar exactly as it does for braço 01.

In [9]:
pooled = emb_steps.pool_by_week(articles, locations, encoder.dim)

print(f"{articles.height} articles → {pooled.height} pooled rows over "
      f"{pooled[EW].n_unique()} weeks × {pooled[LOCATION].n_unique()} locations\n")
pooled

5 articles → 6 pooled rows over 5 weeks × 2 locations



ew,location,emb_mean,n_articles
date,str,"array[f32, 768]",i64
2002-09-29,"""BR""","[-0.011618, -0.005839, … -0.087426]",1
2002-09-29,"""RJ""","[-0.011618, -0.005839, … -0.087426]",1
2008-03-30,"""BR""","[0.010499, -0.00972, … -0.076836]",1
2011-01-09,"""BR""","[-0.034178, -0.001357, … -0.060466]",1
2016-01-31,"""BR""","[-0.01718, -0.007688, … -0.079641]",1
2021-03-14,"""BR""","[-0.019445, -0.024048, … -0.041803]",1


### BR plus each named UF, from one article

The spread is visible on a single fixture page: one article becomes a national row
and one row per state it names, all in the same week.

`‖emb_mean‖` is printed beside the count as a **reading aid** — the mean of unit
vectors is not a unit vector, and how far it falls short is how far apart the week's
articles are. It is deliberately *not* a published column, for the same reason
`emb_std` was removed: as a feature it is `emb_mean` again.

In [10]:
example = pooled.filter(pl.col(EW) == pooled[EW].min())
print(f"week {example[EW][0]}: {example.height} rows from "
      f"{example[N_ARTICLES].max()} article(s)\n")
example.select(
    LOCATION,
    N_ARTICLES,
    pl.Series("mean_norm", np.linalg.norm(example[EMB_MEAN].to_numpy(), axis=1)).round(4),
)

week 2002-09-29: 2 rows from 1 article(s)



location,n_articles,mean_norm
str,i64,f32
"""BR""",1,1.0
"""RJ""",1,1.0


## Integrity checks — fixture tables

In [11]:
for check in embedding_checks(articles, pooled, FU_PATH):
    print(check)

[PASS] article_schema: 8 columns in published order
[PASS] pooled_schema: 4 columns in published order
[PASS] article_unique_keys: 5 rows, all keys distinct
[PASS] pooled_unique_keys: 6 rows, all keys distinct
[PASS] article_weeks_start_on_sunday: all ew dates are Sundays
[PASS] pooled_weeks_start_on_sunday: all ew dates are Sundays
[PASS] locations_known: 2 known locations
[PASS] dim_consistent: 768-wide vectors throughout
[PASS] variant_consistent: bertimbau-mean · neuralmind/bert-base-portuguese-cased
[PASS] finite: all vectors are finite
[PASS] article_counts: every row pools at least one article
[PASS] token_counts: every article tokenized to >= 1 token
[PASS] national_row_per_week: 5 weeks all carry BR


## Optional — the real corpus, and the whole bracket

The fixture is five pages. Truncation at scale, the weekly coverage, the anisotropy
of real embeddings and the spread *across* the bracket only exist in the
4,369-article corpus:

```bash
uv sync --extra embeddings
uv run arboili transform news-embeddings --variant all   # ~6 configurations
```

In [12]:
if bracket_ready():
    real_articles = pl.read_parquet(tables_for(BASE_VARIANT_ID)[0])
    real_pooled = pl.read_parquet(tables_for(BASE_VARIANT_ID)[1])

    print(f"{real_articles.height:,} articles → {real_pooled.height:,} pooled rows")
    print(f"weeks     {real_pooled[EW].min()} → {real_pooled[EW].max()}")
    print(f"locations {real_pooled[LOCATION].n_unique()} "
          f"(BR + {real_pooled[LOCATION].n_unique() - 1} UFs)")
    print(f"model     {real_articles['model_id'][0]}@{real_articles['model_revision'][0][:8]}")

    print("\ntoken count percentiles: " + ", ".join(
        f"p{q}={real_articles[N_TOKENS].quantile(q / 100):.0f}" for q in (50, 75, 90, 99)
    ))
else:
    print("Run `uv run arboili transform news-embeddings --variant all` first")

4,369 articles → 5,152 pooled rows
weeks     2002-02-17 → 2026-09-13
locations 28 (BR + 27 UFs)
model     neuralmind/bert-base-portuguese-cased@94d69c95

token count percentiles: p50=438, p75=637, p90=895, p99=1965


### The bracket, side by side

One row per configuration. `‖mean‖` is the average norm of the weekly centroids:
the closer to 1, the more the week's articles agree with each other. It is a
**reading of the table**, not a feature — for the same reason `emb_std` was removed,
it is a function of `emb_mean`. What it is good for is comparing configurations:
a whitened variant drops it sharply, which is the anisotropy being corrected.

Truncation is counted against **each variant's own window**, not a single 512.

In [13]:
if bracket_ready():
    rows = []
    for variant_id in VARIANT_ORDER:
        settings = VARIANTS[variant_id]
        art = pl.read_parquet(tables_for(variant_id)[0])
        pool = pl.read_parquet(tables_for(variant_id)[1])
        over, longest = emb_steps.truncation_stats(art, settings.max_tokens)
        rows.append({
            "variant": variant_id,
            "encoder": settings.model_name.split("/")[-1].replace("-portuguese", ""),
            "dim": art["dim"][0],
            "pooling": settings.pooling,
            "window": settings.max_tokens,
            "whitening": settings.whitening,
            "articles": art.height,
            "rows": pool.height,
            "trunc %": round(100 * over / art.height, 1),
            "longest": longest,
            "‖mean‖": round(float(np.linalg.norm(pool[EMB_MEAN].to_numpy(), axis=1).mean()), 3),
        })
    bracket = pl.DataFrame(rows)
    print(bracket)
else:
    print("Run `uv run arboili transform news-embeddings --variant all` first")

shape: (6, 11)
┌────────────────────┬────────────────────┬──────┬─────────┬───┬──────┬─────────┬─────────┬────────┐
│ variant            ┆ encoder            ┆ dim  ┆ pooling ┆ … ┆ rows ┆ trunc % ┆ longest ┆ ‖mean‖ │
│ ---                ┆ ---                ┆ ---  ┆ ---     ┆   ┆ ---  ┆ ---     ┆ ---     ┆ ---    │
│ str                ┆ str                ┆ i64  ┆ str     ┆   ┆ i64  ┆ f64     ┆ i64     ┆ f64    │
╞════════════════════╪════════════════════╪══════╪═════════╪═══╪══════╪═════════╪═════════╪════════╡
│ bertimbau-mean     ┆ bert-base-cased    ┆ 768  ┆ mean    ┆ … ┆ 5152 ┆ 39.0    ┆ 7624    ┆ 0.987  │
│ bertimbau-whitenin ┆ bert-base-cased    ┆ 768  ┆ mean    ┆ … ┆ 5152 ┆ 39.0    ┆ 7624    ┆ 0.854  │
│ g                  ┆                    ┆      ┆         ┆   ┆      ┆         ┆         ┆        │
│ e5-large           ┆ multilingual-e5-la ┆ 1024 ┆ mean    ┆ … ┆ 5152 ┆ 40.8    ┆ 8397    ┆ 0.983  │
│                    ┆ rge                ┆      ┆         ┆   ┆      ┆     

### The anisotropy, measured on the real vectors

The claim whitening makes is falsifiable on this corpus: unrelated articles should
be less alike after the transform. The two article tables are already on disk, so
this compares them directly rather than recomputing anything — on a 1,000-article
sample, since the full 4,369 × 4,369 Gram matrix is 19 M pairs to say one number.

In [14]:
if bracket_ready():
    sample = np.random.default_rng(0).choice(
        real_articles.height, size=min(1000, real_articles.height), replace=False
    )
    base_vectors = pl.read_parquet(tables_for("bertimbau-mean")[0])[EMBEDDING].to_numpy()[sample]
    white_vectors = pl.read_parquet(tables_for("bertimbau-whitening")[0])[EMBEDDING].to_numpy()[sample]

    print(f"mean pairwise cosine over {len(sample):,} articles, "
          f"{base_vectors.shape[1]}-wide")
    print(f"  base (masked mean)          = {mean_cosine(base_vectors):+.4f}")
    print(f"  whitened                    = {mean_cosine(white_vectors):+.4f}")
    print("\nA high base value is the cone: the encoder's own geometry, not a finding "
          "about dengue news.")
else:
    print("Run `uv run arboili transform news-embeddings --variant all` first")

mean pairwise cosine over 1,000 articles, 768-wide
  base (masked mean)          = +0.8792
  whitened                    = -0.0002

A high base value is the cone: the encoder's own geometry, not a finding about dengue news.


### Integrity checks — the real tables

Every configuration, including the provenance guard: a table whose `model_id` is not
the model its variant declares fails, which is the failure every other check would
miss.

In [15]:
if bracket_ready():
    for variant_id in VARIANT_ORDER:
        art = pl.read_parquet(tables_for(variant_id)[0])
        pool = pl.read_parquet(tables_for(variant_id)[1])
        failures = [c for c in embedding_checks(art, pool, FU_PATH) if not c.passed]
        status = "PASS" if not failures else "FAIL"
        print(f"[{status}] {variant_id:22s} {art.height:,} articles, {pool.height:,} rows"
              + (f" — {'; '.join(str(c) for c in failures)}" if failures else ""))
else:
    print("Run `uv run arboili transform news-embeddings --variant all` first")

[PASS] bertimbau-mean         4,369 articles, 5,152 rows


[PASS] bertimbau-whitening    4,369 articles, 5,152 rows


[PASS] e5-large               4,369 articles, 5,152 rows


[PASS] bge-m3                 4,369 articles, 5,152 rows


[PASS] title-only             4,369 articles, 5,152 rows
[PASS] serafim-title          4,369 articles, 5,152 rows


## Exploratory — does any of it track SINAN?

**This is not the protocol-00 comparison.** It is a first look at the paper's actual
question, and the caveats are the same ones braço 01's notebook carries: coverage
*follows* the epidemic curve, the news series is national while the target is UF, and
per-UF rows are mention-based.

A 768-wide vector cannot be correlated with a scalar curve, so it is reduced to its
leading principal component — fit by SVD on the national weekly means themselves.
The share of variance PC1 explains is printed next to every correlation: a PC1 that
explains 12% is not a summary of anything, and quoting its correlation without that
number would be a category error.

`n_articles` is in every table as the reference row. It is braço 01's `news_volume`
under another name, and it is what any dense reading has to beat to earn its
dimensions.

SINAN here is `ew_symptom_onset`, the same Sunday calendar, summed to (week, state).

In [16]:
sinan_weekly = (
    pl.scan_parquet(SINAN_PATH)
    .group_by("ew_symptom_onset", "state_abbrev")
    .agg(pl.col("case_count").sum().alias("casos"))
    .collect()
)
sinan_national = (
    sinan_weekly.group_by("ew_symptom_onset")
    .agg(pl.col("casos").sum())
    .rename({"ew_symptom_onset": EW})
)


def national_frame(pooled_frame):
    # Join one braco's national rows onto the case series, on the same calendar.
    return (
        pooled_frame.filter(pl.col(LOCATION) == "BR")
        .join(sinan_national, on=EW, how="inner")
        .sort(EW)
    )


def first_component(matrix):
    # (scores, share of variance explained) for the leading component.
    centred = matrix - matrix.mean(axis=0, keepdims=True)
    u, s, _ = np.linalg.svd(centred, full_matrices=False)
    return (u[:, 0] * s[0]).astype(np.float32), float((s ** 2 / (s ** 2).sum())[0])

In [17]:
if bracket_ready():
    rows = []
    for variant_id in VARIANT_ORDER:
        br = national_frame(pl.read_parquet(tables_for(variant_id)[1]))
        pc1, explained = first_component(br[EMB_MEAN].to_numpy())
        br = br.with_columns(pl.Series("pc1", pc1))
        rows.append({
            "variant": variant_id,
            "weeks": br.height,
            "PC1 var %": round(100 * explained, 1),
            "PC1": round(br.select(pl.corr("pc1", "casos", method="spearman")).item(), 3),
            "n_articles": round(br.select(pl.corr(N_ARTICLES, "casos", method="spearman")).item(), 3),
        })
    print(f"national weeks with both series: {br.height:,}  ({br[EW].min()} → {br[EW].max()})")
    print("PC1 is the leading component of that variant's weekly means; "
          "n_articles is the same column in every row.\n")
    print(pl.DataFrame(rows))
else:
    print("Run `uv run arboili transform news-embeddings --variant all` first")

national weeks with both series: 481  (2013-07-28 → 2026-08-23)
PC1 is the leading component of that variant's weekly means; n_articles is the same column in every row.

shape: (6, 5)
┌─────────────────────┬───────┬───────────┬───────┬────────────┐
│ variant             ┆ weeks ┆ PC1 var % ┆ PC1   ┆ n_articles │
│ ---                 ┆ ---   ┆ ---       ┆ ---   ┆ ---        │
│ str                 ┆ i64   ┆ f64       ┆ f64   ┆ f64        │
╞═════════════════════╪═══════╪═══════════╪═══════╪════════════╡
│ bertimbau-mean      ┆ 481   ┆ 20.8      ┆ 0.119 ┆ 0.347      │
│ bertimbau-whitening ┆ 481   ┆ 1.1       ┆ 0.188 ┆ 0.347      │
│ e5-large            ┆ 481   ┆ 6.9       ┆ 0.243 ┆ 0.347      │
│ bge-m3              ┆ 481   ┆ 7.8       ┆ 0.282 ┆ 0.347      │
│ title-only          ┆ 481   ┆ 9.8       ┆ 0.178 ┆ 0.347      │
│ serafim-title       ┆ 481   ┆ 7.6       ┆ 0.219 ┆ 0.347      │
└─────────────────────┴───────┴───────────┴───────┴────────────┘


### Lead or lag?

If the representation carried early warning, its value at week *t* would correlate
better with cases at *t+k* than at *t*. The table shifts both series and reports the
national Spearman coefficient in each direction, for the base configuration.

In [18]:
if bracket_ready():
    br = national_frame(pl.read_parquet(tables_for(BASE_VARIANT_ID)[1]))
    pc1, explained = first_component(br[EMB_MEAN].to_numpy())
    br = br.with_columns(pl.Series("pc1", pc1))

    scan = []
    for k in range(0, 9):
        shifted = br.with_columns(
            pl.col("casos").shift(-k).alias("casos_futuros"),
            pl.col("pc1").shift(-k).alias("pc1_futuro"),
        )
        scan.append({
            "k (weeks)": k,
            "pc1[t] × cases[t+k]": shifted.select(pl.corr("pc1", "casos_futuros", method="spearman")).item(),
            "cases[t] × pc1[t+k]": shifted.select(pl.corr("casos", "pc1_futuro", method="spearman")).item(),
            "n_articles[t] × cases[t+k]": shifted.select(pl.corr(N_ARTICLES, "casos_futuros", method="spearman")).item(),
        })
    print(pl.DataFrame(scan).with_columns(pl.exclude("k (weeks)").round(3)))
else:
    print("Run `uv run arboili transform news-embeddings --variant all` first")

shape: (9, 4)
┌───────────┬─────────────────────┬─────────────────────┬────────────────────────────┐
│ k (weeks) ┆ pc1[t] × cases[t+k] ┆ cases[t] × pc1[t+k] ┆ n_articles[t] × cases[t+k] │
│ ---       ┆ ---                 ┆ ---                 ┆ ---                        │
│ i64       ┆ f64                 ┆ f64                 ┆ f64                        │
╞═══════════╪═════════════════════╪═════════════════════╪════════════════════════════╡
│ 0         ┆ 0.119               ┆ 0.119               ┆ 0.347                      │
│ 1         ┆ 0.13                ┆ 0.101               ┆ 0.331                      │
│ 2         ┆ 0.133               ┆ 0.108               ┆ 0.319                      │
│ 3         ┆ 0.132               ┆ 0.102               ┆ 0.306                      │
│ 4         ┆ 0.137               ┆ 0.096               ┆ 0.293                      │
│ 5         ┆ 0.118               ┆ 0.083               ┆ 0.279                      │
│ 6         ┆ 0.119          

### Per federative unit

Restricted to UFs with at least 20 overlapping weeks, because a coefficient on
eight points is noise. `semanas` is printed next to every number for that reason.

Read these with Step 6 in mind: the scoping is mention-based, so a UF only appears
in the weeks its name appeared in the news.

In [19]:
if bracket_ready():
    per_uf_pool = pl.read_parquet(tables_for(BASE_VARIANT_ID)[1])
    per_uf = (
        per_uf_pool.filter(pl.col(LOCATION) != "BR")
        .join(
            sinan_weekly.rename({"ew_symptom_onset": EW, "state_abbrev": LOCATION}),
            on=[EW, LOCATION],
            how="inner",
        )
        .sort(EW)
    )
    rows = []
    for (location,), group in per_uf.group_by([LOCATION], maintain_order=True):
        if group.height < 20:
            continue
        pc1, _ = first_component(group[EMB_MEAN].to_numpy())
        group = group.with_columns(pl.Series("pc1", pc1))
        rows.append({
            "location": location,
            "semanas": group.height,
            "PC1": group.select(pl.corr("pc1", "casos", method="spearman")).item(),
            N_ARTICLES: group.select(pl.corr(N_ARTICLES, "casos", method="spearman")).item(),
        })
    print(pl.DataFrame(rows)
          .with_columns(pl.exclude("location", "semanas").round(3))
          .sort("PC1", descending=True))
else:
    print("Run `uv run arboili transform news-embeddings --variant all` first")

shape: (27, 4)
┌──────────┬─────────┬────────┬────────────┐
│ location ┆ semanas ┆ PC1    ┆ n_articles │
│ ---      ┆ ---     ┆ ---    ┆ ---        │
│ str      ┆ i64     ┆ f64    ┆ f64        │
╞══════════╪═════════╪════════╪════════════╡
│ MG       ┆ 137     ┆ 0.342  ┆ 0.26       │
│ RS       ┆ 57      ┆ 0.234  ┆ 0.324      │
│ SE       ┆ 39      ┆ 0.224  ┆ 0.026      │
│ AL       ┆ 32      ┆ 0.205  ┆ 0.168      │
│ PA       ┆ 35      ┆ 0.196  ┆ -0.052     │
│ RR       ┆ 38      ┆ 0.07   ┆ -0.155     │
│ DF       ┆ 112     ┆ 0.064  ┆ 0.496      │
│ RO       ┆ 42      ┆ 0.056  ┆ -0.101     │
│ …        ┆ …       ┆ …      ┆ …          │
│ SC       ┆ 53      ┆ -0.212 ┆ 0.435      │
│ CE       ┆ 67      ┆ -0.224 ┆ 0.141      │
│ MS       ┆ 66      ┆ -0.228 ┆ 0.209      │
│ MT       ┆ 76      ┆ -0.233 ┆ 0.322      │
│ GO       ┆ 76      ┆ -0.245 ┆ 0.219      │
│ PR       ┆ 90      ┆ -0.274 ┆ 0.323      │
│ ES       ┆ 63      ┆ -0.293 ┆ 0.341      │
│ SP       ┆ 265     ┆ -0.329 ┆ 0.377   

## The three braços, side by side

Braços 01 and 03 publish the same grain — (ew, location), BR plus the UFs — so the
three representations can be put on one calendar. Each is reduced to scalars a
correlation can be computed against, and **each reduction is a choice that has to be
read, not a property of the braço**:

| braço | scalars | what they are |
|---|---|---|
| 01 — lexicon | `news_volume`, `news_severity` | counts of matching articles and of severity terms |
| 02 — embeddings | `PC1`, `n_articles` | the leading component of the weekly means, and the count |
| 03 — topics | `PC1`, `top share` | the leading component of the 15-dim share vector, and the largest share |

`n_articles` appears in braços 01 and 02 as the same number under two names; braço
03 carries its own `n_articles` too. That redundancy is the point of the section: a
representation that does not beat the count of articles in the week has not earned
its dimensions.

**This is descriptive and it is not protocol 00.** The protocol's metric is the
forecaster's error under a rolling origin, with a shuffled-text negative control;
none of that has run.

In [20]:
if bracket_ready() and LEXICON_EW_PATH.exists() and TOPICS_EW_PATH.exists():
    base_pool = pl.read_parquet(tables_for(BASE_VARIANT_ID)[1])
    cases = national_frame(base_pool).select(EW, "casos")

    # 01 — the lexicon counts, straight off the table.
    lex = national_frame(pl.read_parquet(LEXICON_EW_PATH)).select(
        EW,
        pl.col("news_volume").cast(pl.Float64),
        pl.col("news_severity").cast(pl.Float64),
    )

    # 02 — the leading component of the weekly means, plus the count.
    emb = national_frame(base_pool)
    emb_pc1, emb_explained = first_component(emb[EMB_MEAN].to_numpy())
    emb = emb.select(
        EW,
        # `astype(np.float64)`, not `astype(pl.Float64)`: numpy does not know
        # polars dtypes and coerces the dtype object into an `object` array,
        # which lands in the frame as an Object column and then makes `pl.corr`
        # fail with "could not determine supertype of: [object, u32]".
        pl.Series("PC1", emb_pc1.astype(np.float64)),
        pl.col(N_ARTICLES).cast(pl.Float64),
    )

    # 03 — the leading component of the 15-dim share vector, plus the largest share.
    # The component is fit on the same 481 weeks the correlation uses, so all
    # three arms are reduced on one basis: fitting it on the full 965-week span
    # and correlating on a 481-week subset would measure a different object.
    topics = (
        pl.read_parquet(TOPICS_EW_PATH)
        .filter(pl.col("location") == "BR")
        .pivot(index=EW, on="topic_id", values="share")
        .sort(EW)
        .fill_null(0.0)
    )
    topics = topics.join(cases, on=EW, how="inner").sort(EW)
    share_cols = [c for c in topics.columns if c not in (EW, "casos")]
    shares = topics.select(share_cols).to_numpy()
    topic_pc1, topic_explained = first_component(shares)
    topics = topics.select(
        EW,
        pl.Series("PC1", topic_pc1.astype(np.float64)),
        pl.Series("top share", shares.max(axis=1).astype(np.float64)),
    )

    print(f"02's PC1 explains {100 * emb_explained:.1f}% of the variance of the weekly means")
    print(f"03's PC1 explains {100 * topic_explained:.1f}% of the variance of the topic shares\n")

    rows = []
    for arm, frame in {"01 lexicon": lex, "02 embeddings": emb, "03 topics": topics}.items():
        joined = frame.join(cases, on=EW, how="inner")
        for name in frame.columns:
            if name == EW:
                continue
            rows.append({
                "arm": arm,
                "scalar": name,
                "weeks": joined.height,
                "spearman vs cases": round(
                    joined.select(pl.corr(name, "casos", method="spearman")).item(), 3
                ),
            })
    print(pl.DataFrame(rows))
else:
    print("Needs the bracket, braço 01's and braço 03's tables — see the section above")

02's PC1 explains 20.8% of the variance of the weekly means
03's PC1 explains 21.7% of the variance of the topic shares

shape: (6, 4)
┌───────────────┬───────────────┬───────┬───────────────────┐
│ arm           ┆ scalar        ┆ weeks ┆ spearman vs cases │
│ ---           ┆ ---           ┆ ---   ┆ ---               │
│ str           ┆ str           ┆ i64   ┆ f64               │
╞═══════════════╪═══════════════╪═══════╪═══════════════════╡
│ 01 lexicon    ┆ news_volume   ┆ 481   ┆ 0.347             │
│ 01 lexicon    ┆ news_severity ┆ 481   ┆ 0.308             │
│ 02 embeddings ┆ PC1           ┆ 481   ┆ 0.119             │
│ 02 embeddings ┆ n_articles    ┆ 481   ┆ 0.347             │
│ 03 topics     ┆ PC1           ┆ 481   ┆ 0.32              │
│ 03 topics     ┆ top share     ┆ 481   ┆ -0.1              │
└───────────────┴───────────────┴───────┴───────────────────┘


## What the exploratory section shows

Run on 2026-10-14, over the 481 national weeks where both series exist
(2013-07-28 → 2026-08-23), with the whole bracket on disk. Five readings, and the
first one is the one the bracket was built for.

- **The bracket spans a factor of 2.4, and none of it reaches the count.** PC1
  correlates +0.119 (`bertimbau-mean`), +0.178 (`title-only`), +0.188 (whitened),
  +0.219 (`serafim-title`), +0.243 (`e5-large`), +0.282 (`bge-m3`) — against
  **+0.347** for `n_articles`, which is the same column in every row and is braço
  01's `news_volume` under another name. "Dense embeddings correlate +0.12 with
  cases" would have been a sentence about one encoder and one pooling; BGE-M3
  nearly doubles it.
- **Reading the body hurt.** `title-only` beats the full-text base (+0.178 against
  +0.119) while truncating nothing at all — its longest title is 35 tokens. So the
  512-token window is not what limits the base configuration, and the usual excuse
  for a weak dense feature ("the articles are too long") is not available here.
- **Whitening fixed the geometry and not the correlation.** Mean pairwise cosine
  over a 1,000-article sample falls from **+0.8792** to **−0.0002**: the cone is
  gone, exactly as [Ethayarajh 2019](https://doi.org/10.18653/v1/D19-1006)
  describes it. But spreading the variance evenly removes the dominant direction
  too — PC1 drops from 20.8% of the variance to **1.1%** — so its +0.188 is the
  correlation of a direction that is no longer a summary of anything. A single
  scalar reading is the wrong instrument for that variant.
- **Braço 03's leading component (+0.320) beats every braço-02 variant**, and
  braço 01's `news_severity` (+0.308) beats all but one. On this descriptive
  metric the dense representation is the weakest of the three, not the strongest.
- **Contemporaneous, not leading.** The lead/lag table above is flat in both
  directions for every scalar. Nothing here warns ahead, which matches braço 01
  and the framing decision in [`notes/README.md`](../notes/README.md): coverage
  measures attention, not the epidemic's future.

The uncomfortable reading is not "the embeddings lost". It is that the two
variants that gain most over the base are **the two whose pretraining cutoff is
undocumented**, so the improvement cannot be separated from leakage by this
notebook. Protocol 00's temporal validation is what decides that, and it has not
run.

None of this is the protocol-00 comparison. That comparison — braços 01, 02 and 03
under one model and one set of metrics, with the negative control — has not run, and
these numbers must not be quoted as its result.

## Running it for real

This notebook mirrors `transform()` in
`src/pipeline/news/embeddings/transform.py`:

```bash
uv sync --extra embeddings                    # torch + transformers (~800 MB), once
uv run arboili transform news-embeddings --variant all
```

Writes two tables per configuration: `EBC_dengue_embeddings[_<variant>]_article.parquet`
(one row per article — the expensive half) and `..._EW.parquet` (the pooled weekly
table the forecaster consumes). The base configuration keeps the unsuffixed names.
Parquet only: a 768-float column does not survive csv text form.

Re-pooling from the article table is seconds. Re-encoding is the whole run, which is
why both are written.

## What this notebook does not do

- **No fine-tuning.** The encoders are frozen, by the design `notes/02` fixes.
  Fine-tuning on relevance classification is a separate sub-experiment and would
  change the braço's nature — decide it before comparing, not during.
- **No protocol-00 comparison.** The three-arm section above is descriptive; the
  protocol's metric is forecast error under a rolling origin.
- **No negative control.** Shuffling the text and re-running is what protocol 00
  requires to prove the representation is being used at all — Tan 2024
  (arXiv:2406.16964) is the reason that control exists.
- **No whitening fit window.** The whitened variant here is fitted on the whole
  corpus, which is right for a descriptive reading and wrong for a predictive one.
  `whitening_fit_until=` is the argument the rolling-origin run must pass.
- **No dimensionality reduction for the forecaster.** PC1 above is for reading, not
  for feeding; whether the 768-wide vector is used whole or reduced (and the variance
  that keeps) is a modelling decision still open in `notes/02`.
- **No chunked pooling.** Long articles are truncated at each variant's window.
  Whether the tail matters is an open question this notebook measures the size of,
  not one it answers.